# Prompt sensitivity experiment: model size comparison (Kaggle)
**Before running:** Settings (right panel) -> *Accelerator* = **GPU T4 x2**, *Internet* = **On**.
Add your uploaded dataset (with `sample.csv` and the `.py` scripts) via **Add Input**.

Run the cells in order. Results are saved in `/kaggle/working/` (Output tab).
If the session stops, just run everything again: finished answers are kept (`--resume`).

## 1. Record the hardware (for your Methods section)

In [2]:
!nvidia-smi
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv > /kaggle/working/hardware.txt
!cat /kaggle/working/hardware.txt

Fri Nov 20 21:50:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Copy your scripts and sample into the working folder

In [3]:
import glob, shutil, os
os.chdir('/kaggle/working')
found = glob.glob('/kaggle/input/**/sample.csv', recursive=True)
assert found, 'sample.csv not found - did you add your dataset with Add Input?'
src = os.path.dirname(found[0])
for f in ['sample.csv', 'prompts.py', 'run_experiment.py', 'analyze.py']:
    shutil.copy(os.path.join(src, f), '/kaggle/working/')
print('Copied from', src); print(sorted(os.listdir('.')))

Copied from /kaggle/input/datasets/dovydasgan/prompt-sensitivity
['.virtual_documents', 'analyze.py', 'hardware.txt', 'prompts.py', 'run_experiment.py', 'sample.csv']


## 3. Install and start Ollama

In [9]:
!apt-get -qq update && apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, requests
server = subprocess.Popen(['ollama', 'serve'], stdout=open('/kaggle/working/ollama.log', 'w'),
                          stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        requests.get('http://localhost:11434/api/tags', timeout=2); print('Ollama is running'); break
    except Exception:
        time.sleep(2)
!ollama --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%####################                                              39.3%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Ollama is running
ollama version is 0.40.16


In [10]:
!ollama pull qwen2.5:32b
!ollama list

[download progress output removed to reduce file size]
NAME           ID              SIZE      MODIFIED      
qwen2.5:32b    9f13ba1299af    19 GB     5 seconds ago    
qwen2.5:14b    7cdf5a0187d5    9.0 GB    7 minutes ago    
qwen2.5:7b     845dbda0ea48    4.7 GB    8 minutes ago    


In [5]:
!grep -i -E "cuda|gpu|vram" /kaggle/working/ollama.log | head -20

time=2026-11-20T21:51:16.748Z level=INFO source=routes.go:2474 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_CREATE_REMOTE:false OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_IGPU_ENABLE: OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https:

## 4. Download the models
Same family (Qwen 2.5), three sizes. `qwen2.5:7b` is included again so that all sizes run on the *same* machine.
The 32B model is about 20 GB; if it does not fit or is too slow, remove it from `MODELS`.

In [11]:
MODELS = ['qwen2.5:7b', 'qwen2.5:14b', 'qwen2.5:32b']
for m in MODELS:
    !ollama pull {m}
!ollama list

pulling manifest ⠋ pulling manifest ⠙ pulling manifest 
pulling 2bada8a74506: 100% ▕██████████████████▏ 4.7 GB                         
pulling 66b9ea09bd5b: 100% ▕██████████████████▏   68 B                         
pulling eb4402837c78: 100% ▕██████████████████▏ 1.5 KB                         
pulling 832dd9e00a68: 100% ▕██████████████████▏  11 KB                         
pulling 2f15b3218f05: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
removing unused layers 
success 
pulling manifest ⠋ pulling manifest ⠙ pulling manifest 
pulling 2049f5674b1e: 100% ▕██████████████████▏ 9.0 GB                         
pulling 66b9ea09bd5b: 100% ▕██████████████████▏   68 B                         
pulling eb4402837c78: 100% ▕██████████████████▏ 1.5 KB                         
pulling 832dd9e00a68: 100% ▕██████████████████▏  11 KB                         
pulling db59b814cab7: 100% ▕██████████████████▏  488 B                         
verifying sh

In [8]:
!df -h /root

Filesystem      Size  Used Avail Use% Mounted on
overlay         8.0T  5.6T  2.5T  70% /


## 5. Quick pilot (2 posts per model)
Check that labels look right, then run `ollama ps` to see GPU use (PROCESSOR should say 100% GPU).

In [12]:
for m in MODELS:
    tag = m.replace(':', '_').replace('.', '')
    !python run_experiment.py --models {m} --limit 2 --output pilot_kaggle_{tag}.csv --resume
!ollama ps
import pandas as pd
for f in sorted(glob.glob('pilot_kaggle_*.csv')):
    d = pd.read_csv(f); print(f, '| mean seconds:', round(d.seconds.iloc[1:].mean(), 2))
    print(d[['variant', 'gold', 'raw_answer', 'label']].head(8).to_string(), '\n')

Running 24 prompts ...
  24/24
Done. Results in pilot_kaggle_qwen25_7b.csv, settings in pilot_kaggle_qwen25_7b_metadata.json
Running 24 prompts ...
  24/24
Done. Results in pilot_kaggle_qwen25_14b.csv, settings in pilot_kaggle_qwen25_14b_metadata.json
Running 24 prompts ...
  24/24
Done. Results in pilot_kaggle_qwen25_32b.csv, settings in pilot_kaggle_qwen25_32b_metadata.json
NAME           ID              SIZE      PROCESSOR          CONTEXT    RUNNER      UNTIL              
qwen2.5:32b    9f13ba1299af    20 GB     30%/70% CPU/GPU    2048       llamacpp    4 minutes from now    
qwen2.5:14b    7cdf5a0187d5    9.1 GB    100% GPU           2048       llamacpp    2 minutes from now    
qwen2.5:7b     845dbda0ea48    4.6 GB    100% GPU           2048       llamacpp    2 minutes from now    
pilot_kaggle_qwen25_14b.csv | mean seconds: 0.13
     variant      gold raw_answer     label
0   v1_plain  positive   positive  positive
1   v1_plain  positive   positive  positive
2   v1_plain  posit

In [13]:
!ollama ps

NAME           ID              SIZE     PROCESSOR          CONTEXT    RUNNER      UNTIL                   
qwen2.5:32b    9f13ba1299af    20 GB    30%/70% CPU/GPU    2048       llamacpp    About a minute from now    


## 6. Full run
One results file per model. Estimated time = 3,600 x mean seconds from the pilot.
If the session stops, rerun this cell: `--resume` continues where it stopped.

In [14]:
import subprocess
for m in MODELS:
    for other in MODELS:
        subprocess.run(['ollama', 'stop', other])   # free the GPUs
    tag = m.replace(':', '_').replace('.', '')
    !python run_experiment.py --models {m} --repeats 3 --output results_kaggle_{tag}.csv --resume

Running 3600 prompts ...
  25/3600
  50/3600
  75/3600
  100/3600
  125/3600
  150/3600
  175/3600
  200/3600
  225/3600
  250/3600
  275/3600
  300/3600
  325/3600
  350/3600
  375/3600
  400/3600
  425/3600
  450/3600
  475/3600
  500/3600
  525/3600
  550/3600
  575/3600
  600/3600
  625/3600
  650/3600
  675/3600
  700/3600
  725/3600
  750/3600
  775/3600
  800/3600
  825/3600
  850/3600
  875/3600
  900/3600
  925/3600
  950/3600
  975/3600
  1000/3600
  1025/3600
  1050/3600
  1075/3600
  1100/3600
  1125/3600
  1150/3600
  1175/3600
  1200/3600
  1225/3600
  1250/3600
  1275/3600
  1300/3600
  1325/3600
  1350/3600
  1375/3600
  1400/3600
  1425/3600
  1450/3600
  1475/3600
  1500/3600
  1525/3600
  1550/3600
  1575/3600
  1600/3600
  1625/3600
  1650/3600
  1675/3600
  1700/3600
  1725/3600
  1750/3600
  1775/3600
  1800/3600
  1825/3600
  1850/3600
  1875/3600
  1900/3600
  1925/3600
  1950/3600
  1975/3600
  2000/3600
  2025/3600
  2050/3600
  2075/3600
  2100/3600
  2125/36

⠙ ⠙ ⠹ ⠸ ⠼ ⠴ ⠧ ⠧ ⠇ ⠏ ⠙ ⠹ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ 

Running 3600 prompts ...
  25/3600
  50/3600
  75/3600
  100/3600
  125/3600
  150/3600
  175/3600
  200/3600
  225/3600
  250/3600
  275/3600
  300/3600
  325/3600
  350/3600
  375/3600
  400/3600
  425/3600
  450/3600
  475/3600
  500/3600
  525/3600
  550/3600
  575/3600
  600/3600
  625/3600
  650/3600
  675/3600
  700/3600
  725/3600
  750/3600
  775/3600
  800/3600
  825/3600
  850/3600
  875/3600
  900/3600
  925/3600
  950/3600
  975/3600
  1000/3600
  1025/3600
  1050/3600
  1075/3600
  1100/3600
  1125/3600
  1150/3600
  1175/3600
  1200/3600
  1225/3600
  1250/3600
  1275/3600
  1300/3600
  1325/3600
  1350/3600
  1375/3600
  1400/3600
  1425/3600
  1450/3600
  1475/3600
  1500/3600
  1525/3600
  1550/3600
  1575/3600
  1600/3600
  1625/3600
  1650/3600
  1675/3600
  1700/3600
  1725/3600
  1750/3600
  1775/3600
  1800/3600
  1825/3600
  1850/3600
  1875/3600
  1900/3600
  1925/3600
  1950/3600
  1975/3600
  2000/3600
  2025/3600
  2050/3600
  2075/3600
  2100/3600
  2125/36

⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ 

Running 3600 prompts ...
  25/3600
  50/3600
  75/3600
  100/3600
  125/3600
  150/3600
  175/3600
  200/3600
  225/3600
  250/3600
  275/3600
  300/3600
  325/3600
  350/3600
  375/3600
  400/3600
  425/3600
  450/3600
  475/3600
  500/3600
  525/3600
  550/3600
  575/3600
  600/3600
  625/3600
  650/3600
  675/3600
  700/3600
  725/3600
  750/3600
  775/3600
  800/3600
  825/3600
  850/3600
  875/3600
  900/3600
  925/3600
  950/3600
  975/3600
  1000/3600
  1025/3600
  1050/3600
  1075/3600
  1100/3600
  1125/3600
  1150/3600
  1175/3600
  1200/3600
  1225/3600
  1250/3600
  1275/3600
  1300/3600
  1325/3600
  1350/3600
  1375/3600
  1400/3600
  1425/3600
  1450/3600
  1475/3600
  1500/3600
  1525/3600
  1550/3600
  1575/3600
  1600/3600
  1625/3600
  1650/3600
  1675/3600
  1700/3600
  1725/3600
  1750/3600
  1775/3600
  1800/3600
  1825/3600
  1850/3600
  1875/3600
  1900/3600
  1925/3600
  1950/3600
  1975/3600
  2000/3600
  2025/3600
  2050/3600
  2075/3600
  2100/3600
  2125/36

## 7. Combine and analyse

In [15]:
files = sorted(glob.glob('results_kaggle_*.csv'))
all_res = pd.concat([pd.read_csv(f) for f in files])
print(all_res.groupby('model').size())
all_res.to_csv('results_kaggle.csv', index=False)
!python analyze.py --input results_kaggle.csv

model
qwen2.5:14b    3600
qwen2.5:32b    3600
qwen2.5:7b     3600
dtype: int64

Accuracy per wording (share correct):
variant      v1_plain  v2_formal  v3_casual  v4_reordered
model                                                    
qwen2.5:14b     0.766      0.760      0.709         0.716
qwen2.5:32b     0.723      0.743      0.722         0.757
qwen2.5:7b      0.740      0.733      0.697         0.749

Stability summary:
      model  repeat_consistency  wording_consistency  accuracy_min  accuracy_max  accuracy_spread  invalid_rate  mean_seconds
qwen2.5:14b               0.996                0.797         0.709         0.766            0.057           0.0          0.18
qwen2.5:32b               0.998                0.853         0.722         0.757            0.034           0.0          0.29
 qwen2.5:7b               0.999                0.790         0.697         0.749            0.052           0.0          0.09

Items whose label changed across wordings: 168
      model  item_id

## 8. Download
Open the **Output** tab (or `/kaggle/working`) and download `results_kaggle*.csv`, `*_metadata.json`, `hardware.txt` and the `summary_*.csv` files.